# Seeded Batch Exploration for ModPlant-OPT

## Overview

This notebook is the batch-execution companion to `ModPlant-OPT.ipynb`. Instead of reproducing the fixed HC10-HC40 reference case, it extracts the executable code from the main planning notebook, switches the plant configuration to seeded randomized generation, and evaluates a user-defined seed range in parallel.

The batch workflow is intended for supplementary exploration and for reproducing deterministic infeasible cases such as the capability-mismatch scenario discussed in the repository documentation. The notebook therefore serves as the repository's seed-sweep and throughput-oriented counterpart to the paper-facing reference notebook.


## Notebook Roadmap

The notebook follows the same paper-companion style as the main planner notebook, but focuses on scalable seeded execution.

1. [Runtime Environment and Batch Scope](#Runtime-Environment-and-Batch-Scope)
2. [Notebook Session Setup](#Notebook-Session-Setup)
3. [Batch Configuration and Repository Discovery](#Batch-Configuration-and-Repository-Discovery)
4. [Worker Script Extraction](#Worker-Script-Extraction)
5. [Resource Guard and Per-Seed Worker Execution](#Resource-Guard-and-Per-Seed-Worker-Execution)
6. [Batch Orchestration and Logging](#Batch-Orchestration-and-Logging)


## Runtime Environment and Batch Scope

This notebook assumes the repository root as the working directory and reuses the currently active Python interpreter for worker subprocesses.

- The main planning logic remains defined in `ModPlant-OPT.ipynb`.
- The batch notebook extracts code cells from the main notebook, strips notebook magics, and replaces `use_original_modplants = True` with `False` so that seeded randomized generation becomes active.
- The seed range can be controlled either by editing the notebook constants or by setting `BATCH_SEED_START` and `BATCH_SEED_MAX` in the environment.
- `SAVE_RECIPE_TO_LOCAL` controls whether each worker writes generated XML and parsed JSON recipe artifacts into the repository-local `Recipe/XML` and `Recipe/Json` folders during seeded runs.
- Runtime outputs are written into `Log/` and into the same result folders used by the main notebook workers.


## Notebook Session Setup

The next cell resets the interactive namespace and enables autoreload so local helper edits can be picked up during iterative development.


In [ ]:
%reset -f
%load_ext autoreload
%autoreload 2


## Batch Configuration and Repository Discovery

The following cell imports the orchestration dependencies, defines the core batch parameters, resolves the seed range from notebook constants and optional environment variables, and verifies that `ModPlant-OPT.ipynb` is available in the repository root.

The exposed configuration parameters are:

- `KERN_NUM`: maximum number of concurrent manager threads, each supervising one worker subprocess;
- `MEM_SAFE_THRESHOLD`: high-water mark for pausing launches when RAM usage becomes excessive;
- `TIMEOUT_S`: per-seed timeout;
- `START_INDEX`, `END_INDEX`: default inclusive seed range used when no environment override is present; and
- `SAVE_RECIPE_TO_LOCAL`: switch controlling whether worker runs keep local recipe XML/JSON artifacts under `Recipe/`.


In [ ]:
import concurrent.futures
import gc
import json
import os
import queue
import subprocess
import sys
import threading
import time
from pathlib import Path

import psutil

KERN_NUM = 8
MEM_SAFE_THRESHOLD = 90.0
TIMEOUT_S = 600
START_INDEX = 10
END_INDEX = 15
SAVE_RECIPE_TO_LOCAL = False

print("Python version:", sys.version)
print("HPC Mode: Enabled")
print(f"Target Core/Worker Count: {KERN_NUM}")
print(f"SAVE_RECIPE_TO_LOCAL = {SAVE_RECIPE_TO_LOCAL}")

LOG_DIR = Path("Log")
LOG_DIR.mkdir(parents=True, exist_ok=True)

env_max = os.getenv('BATCH_SEED_MAX')
env_start = os.getenv('BATCH_SEED_START')

arg_max = None
arg_start = None
is_jupyter = 'ipykernel' in sys.modules

if not is_jupyter:
    if len(sys.argv) >= 2 and sys.argv[1].isdigit():
        arg_max = sys.argv[1]
    if len(sys.argv) >= 3 and sys.argv[2].isdigit():
        arg_start = sys.argv[2]
else:
    print("[Info] Jupyter environment detected. Ignoring CLI arguments (sys.argv).")

max_seed = int(arg_max or env_max or END_INDEX)
start_seed = int(arg_start or env_start or START_INDEX)

if start_seed > max_seed:
    raise SystemExit(f'start_seed ({start_seed}) cannot exceed max_seed ({max_seed})')

ROOT = Path('.').resolve()
NB_CANDIDATES = [ROOT / 'ModPlant-OPT.ipynb']
NB_PATH = next((p for p in NB_CANDIDATES if p.exists()), None)

if NB_PATH is None:
    raise FileNotFoundError('Cannot locate ModPlant-OPT.ipynb in the repository root.')

TEMP_SCRIPT_PATH = Path('_temp_worker_script_hpc.py')


## Worker Script Extraction

This stage converts the main notebook into a temporary Python worker script. The extraction is deliberately conservative: it preserves the original execution order of the planning notebook, removes notebook-only magics, activates randomized plant generation, and appends a structured `__WORKER_RESULT__` payload that the batch manager can parse without guessing from free-form console output.

Recipe persistence is controlled through the environment variable `MODPLANT_SAVE_RECIPE_LOCAL`, which the batch worker receives from `SAVE_RECIPE_TO_LOCAL` during subprocess launch.


In [ ]:
def create_worker_script(nb_path, script_path):
    print(f"Extracting code from {nb_path} to {script_path}...")
    try:
        nb_content = nb_path.read_text(encoding='utf-8')
        nb = json.loads(nb_content)
    except Exception as e:
        print(f"Error reading notebook: {e}")
        sys.exit(1)

    code_lines = []
    code_lines.append("import os, sys, json, traceback")
    code_lines.append("if hasattr(sys.stdout, 'reconfigure'):")
    code_lines.append("    sys.stdout.reconfigure(line_buffering=True)")
    code_lines.append("    sys.stderr.reconfigure(line_buffering=True)")

    for cell in nb.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        source = ''.join(cell.get('source', ''))
        clean_source = [line for line in source.split('\n') if not line.lstrip().startswith(('%', '!'))]
        source_text = '\n'.join(clean_source)
        source_text = source_text.replace('use_original_modplants = True', 'use_original_modplants = False')
        if source_text.strip():
            code_lines.append(source_text + '\n')

    footer = """
try:
    _res_feasible = locals().get('bfs_feasible', None)
    _res_path = locals().get('corpus_path', None)
    if _res_path:
        _res_path = str(_res_path)
    print(f"\\n__WORKER_RESULT__:{json.dumps({'status': 'OK', 'feasible': _res_feasible, 'corpus_path': _res_path}, ensure_ascii=False)}", flush=True)
except Exception as e:
    print(f"\\n__WORKER_RESULT__:{json.dumps({'status': f'ERROR: {e}', 'feasible': None, 'corpus_path': None})}", flush=True)
"""
    code_lines.append(footer)

    with open(script_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(code_lines))


create_worker_script(NB_PATH, TEMP_SCRIPT_PATH)


## Resource Guard and Per-Seed Worker Execution

The next cell defines two operational layers.

First, `wait_for_memory()` delays new launches when the machine is already under heavy RAM pressure. Second, `run_single_seed(seed)` executes one deterministic seeded worker process, captures its mixed stdout/stderr stream into a per-seed log file, extracts the structured completion payload when present, enforces the timeout policy, and writes a compact JSON summary for downstream inspection.


In [ ]:
def wait_for_memory():
    """Block while memory usage is above the configured high-water mark."""
    while True:
        mem = psutil.virtual_memory()
        if mem.percent < MEM_SAFE_THRESHOLD:
            return
        print(f"[System] RAM usage high ({mem.percent}%). Waiting for resources...", end='\r')
        time.sleep(2)


def run_single_seed(seed):
    wait_for_memory()
    start_time = time.perf_counter()

    env = os.environ.copy()
    env['MODPLANT_SEED'] = str(seed)
    env['MODPLANT_SAVE_RECIPE_LOCAL'] = '1' if SAVE_RECIPE_TO_LOCAL else '0'
    env['PYTHONUNBUFFERED'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'

    log_txt_path = LOG_DIR / f"seed_{seed}.log"

    status = 'UNKNOWN'
    bfs_feasible = None
    corpus_path = None

    def _reader_thread(pipe, q):
        try:
            for line in iter(pipe.readline, ''):
                q.put(line)
        finally:
            q.put(None)

    try:
        with open(log_txt_path, 'w', encoding='utf-8') as log_file:
            process = subprocess.Popen(
                [sys.executable, '-u', str(TEMP_SCRIPT_PATH)],
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding='utf-8',
                bufsize=1,
            )

            q = queue.Queue()
            reader = threading.Thread(target=_reader_thread, args=(process.stdout, q), daemon=True)
            reader.start()

            timed_out = False

            while True:
                if time.perf_counter() - start_time > TIMEOUT_S:
                    timed_out = True
                    status = f"TIMEOUT (> {TIMEOUT_S}s)"
                    try:
                        process.terminate()
                        process.wait(timeout=2)
                    except Exception:
                        try:
                            process.kill()
                        except Exception:
                            pass
                    break

                try:
                    line = q.get(timeout=0.2)
                except queue.Empty:
                    line = None

                if line is None:
                    if process.poll() is not None:
                        break
                    continue

                log_file.write(line)

                if "__WORKER_RESULT__:" in line:
                    try:
                        json_str = line.split("__WORKER_RESULT__:", 1)[1].strip()
                        data = json.loads(json_str)
                        status = data.get('status', 'OK')
                        bfs_feasible = data.get('feasible')
                        corpus_path = data.get('corpus_path')
                    except Exception:
                        pass

            if not timed_out:
                process.wait()
                if process.returncode != 0 and status == 'UNKNOWN':
                    status = f"CRASH (Exit {process.returncode})"

    except Exception as e:
        status = f"FAILED: {e}"

    duration = time.perf_counter() - start_time
    json_data = {
        'seed': seed,
        'status': status,
        'feasible': bfs_feasible,
        'corpus_path': corpus_path,
        'time_s': duration,
    }

    safe_f = str(bfs_feasible).lower() if bfs_feasible is not None else "none"
    try:
        with open(LOG_DIR / f"seed_{seed}_feasible_{safe_f}.json", 'w', encoding='utf-8') as f:
            json.dump(json_data, f, indent=2)
    except Exception:
        pass

    symbol = '✅' if bfs_feasible else ('❌' if bfs_feasible is False else '❌')
    return f"[Seed {seed}] {symbol} Time: {duration:.1f}s | RAM: {psutil.virtual_memory().percent}%"


## Batch Orchestration and Logging

The final cell launches the batch manager. A `ThreadPoolExecutor` is used only as an orchestration layer because the heavy computation happens inside subprocesses running the extracted planner script. As seeds finish, the notebook reports concise progress lines, triggers periodic garbage collection in the manager process, and removes the temporary worker script at shutdown.


In [ ]:
if __name__ == "__main__":
    print("-" * 50)
    print("Starting HPC Batch Runner")
    print(f"Range: {start_seed} -> {max_seed}")
    print(f"Max Parallel Workers: {KERN_NUM}")
    print(f"Logs: {LOG_DIR.resolve()}")
    print("-" * 50)

    total_start_time = time.perf_counter()

    with concurrent.futures.ThreadPoolExecutor(max_workers=KERN_NUM) as executor:
        future_to_seed = {
            executor.submit(run_single_seed, seed): seed
            for seed in range(start_seed, max_seed + 1)
        }

        finished_count = 0
        total_tasks = len(future_to_seed)

        try:
            for future in concurrent.futures.as_completed(future_to_seed):
                seed = future_to_seed[future]
                try:
                    result_msg = future.result()
                    finished_count += 1
                    print(f"[{finished_count}/{total_tasks}] {result_msg}")
                except Exception as exc:
                    print(f"[Seed {seed}] Exception: {exc}")

                if finished_count % 10 == 0:
                    gc.collect()
        except KeyboardInterrupt:
            print("\nStopping manager... (Ctrl+C pressed)")
            executor.shutdown(wait=False, cancel_futures=True)

    print("-" * 50)
    print(f"All seeds processed in {time.perf_counter() - total_start_time:.2f}s")

    if TEMP_SCRIPT_PATH.exists():
        try:
            TEMP_SCRIPT_PATH.unlink()
        except Exception:
            pass
